In [ ]:
import sys
sys.path.append('..')

import torch
import math
import os
from transformers import AutoTokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset

# Baseline DeepSeekMoE
from source.deepseek_baseline.config import DeepseekConfig as BaselineConfig
from source.deepseek_baseline.model import DeepseekForCausalLM as BaselineModel

# DYNMoE baseline (pure DYNMoE architecture)
from source.DYNMoe_baseline.config import DynMoEConfig as DYNMoEBaseConfig
from source.DYNMoe_baseline.model import DynMoEForCausalLM as DYNMoEBaseModel

# Prototype: DeepSeekMoE with DYNMoE routing
from source.deepseek_dynamics_routing.config import DeepseekConfig as DynmoeConfig
from source.deepseek_dynamics_routing.model import DeepseekForCausalLM as DynmoeModel
from source.deepseek_dynamics_routing.adaptive_tuning import AdaptiveExpertTuningCallback

# Utilities
from source.training_utils.monitoring import ResourceMonitorCallback, MoEMetricsCallback
from source.training_utils.save_model import save_model_and_tokenizer
from source.training_utils.summarization import print_training_summary
from source.data_preprocessing import load_and_preprocess_multiwoz

In [ ]:
# Fine‑tuning hyperparameters
MAX_SEQ_LEN = 256
PER_DEVICE_BATCH = 4          # Reduced for stability
GRAD_ACCUM = 16               # Effective batch = 4 * 2 GPUs * 16 = 128
LEARNING_RATE = 5e-5
NUM_EPOCHS_FT = 3
WARMUP_STEPS = 50
WEIGHT_DECAY = 0.01
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.005

# Paths to pre‑trained models (output from training comparison)
PRETRAINED_BASELINE = "./checkpoints/baseline"
PRETRAINED_DYNMOE_BASE = "./checkpoints/dynmoe_baseline"
PRETRAINED_DYNMOE_ROUTING = "./checkpoints/dynmoe_routing"

# Output directories for fine‑tuned versions
OUTPUT_FT_BASELINE = "./checkpoints/baseline-ft"
OUTPUT_FT_DYNMOE_BASE = "./checkpoints/dynmoe_baseline-ft"
OUTPUT_FT_DYNMOE_ROUTING = "./checkpoints/dynmoe_routing-ft"

# DeepSpeed config (same as used in fine‑tuning example)
ds_config = {
    "train_batch_size": "auto",
    "train_micro_batch_size_per_gpu": "auto",
    "gradient_accumulation_steps": "auto",
    "fp16": {
        "enabled": True,
        "loss_scale": 0,
        "initial_scale_power": 16,
        "hysteresis": 2,
        "min_loss_scale": 1
    },
    "zero_optimization": {
        "stage": 3,
        "offload_optimizer": {"device": "cpu", "pin_memory": True},
        "offload_param": {"device": "cpu", "pin_memory": True},
        "overlap_comm": True,
        "contiguous_gradients": True,
        "reduce_bucket_size": "auto",
        "stage3_prefetch_bucket_size": "auto",
        "stage3_param_persistence_threshold": "auto",
        "stage3_max_live_parameters": 1e9,
        "stage3_max_reuse_distance": 1e9,
        "stage3_gather_16bit_weights_on_model_save": True
    },
    "gradient_clipping": 0.5,          # Reduced for fine‑tuning
    "steps_per_print": 10,
    "wall_clock_breakdown": False,
    "zero_allow_untested_optimizer": True
}

In [ ]:
# Load dataset
torch.cuda.empty_cache()
world_size = torch.cuda.device_count()
print(f"Number of GPUs: {world_size}")

train_sequences, val_sequences, test_sequences = load_and_preprocess_multiwoz(
    zip_path="MultiWOZ-coref/MultiWOZ2_3.zip",
    sample_size=300,
    random_seed=42
)
print(f"Train sequences: {len(train_sequences)}")
print(f"Validation sequences: {len(val_sequences)}")
print(f"Test sequences: {len(test_sequences)}")

In [ ]:
# Tokenizer 
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-moe-16b-base", use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Tokenize dataset
dataset = load_dataset("text", data_files={"train": "train_sequences.txt", "validation": "val_sequences.txt"})

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
        return_attention_mask=True,
    )

tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    num_proc=2,
    desc="Tokenizing datasets"
)
print(f"Train samples: {len(tokenized_datasets['train'])}")
print(f"Validation samples: {len(tokenized_datasets['validation'])}")

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    pad_to_multiple_of=8
)


In [ ]:
# Fine‑tuning function
def fine_tune_model(ModelClass, ConfigClass, pretrained_path, output_dir, is_dynmoe):
    print(f"\n{'='*60}")
    print(f"FINE‑TUNING: {ModelClass.__name__} from {pretrained_path}")
    print(f"{'='*60}")

    # Load pre‑trained model
    model = ModelClass.from_pretrained(pretrained_path)
    model = model.to("cuda:0")
    model.config.use_cache = False
    model.train()

    # Optionally enable gradient checkpointing 
    if hasattr(model, 'gradient_checkpointing_enable'):
        model.gradient_checkpointing_enable()

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    # Training arguments for fine‑tuning
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=NUM_EPOCHS_FT,
        per_device_train_batch_size=PER_DEVICE_BATCH,
        per_device_eval_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=WARMUP_STEPS,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        fp16=True,
        logging_steps=10,
        save_strategy="epoch",
        evaluation_strategy="epoch",
        load_best_model_at_end=False,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="none",
        deepspeed=ds_config,
        ddp_find_unused_parameters=False if world_size > 1 else None,
        max_grad_norm=0.5,
        gradient_checkpointing=False,
        dataloader_num_workers=2,
        remove_unused_columns=True,
        optim="adamw_torch",
        logging_dir=f"{output_dir}/logs",
        seed=42,
        save_only_model=True
    )

    # Callbacks
    resource_monitor = ResourceMonitorCallback()
    moemetrics = MoEMetricsCallback(
        tokenized_datasets["validation"],
        tokenizer,
        data_collator,
        early_stop_patience=EARLY_STOPPING_PATIENCE,
        early_stop_threshold=EARLY_STOPPING_THRESHOLD
    )
    callbacks = [resource_monitor, moemetrics]
    if is_dynmoe:
        adaptive_callback = AdaptiveExpertTuningCallback(audit_steps=10)
        callbacks.append(adaptive_callback)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        data_collator=data_collator,
        tokenizer=tokenizer,
        callbacks=callbacks
    )

    print("Starting fine‑tuning...")
    train_result = trainer.train()
    print("Fine‑tuning finished.")

    eval_results = trainer.evaluate()
    final_loss = eval_results.get("eval_loss", float('inf'))
    perplexity = math.exp(final_loss) if 0 < final_loss < 30 else float('inf')
    print(f"Final validation loss: {final_loss:.4f}")
    print(f"Validation Perplexity: {perplexity:.2f}")

    # Save the fine‑tuned model
    final_output_dir = os.path.join(output_dir, "final")
    unwrapped = trainer.model.module if hasattr(trainer.model, 'module') else trainer.model
    unwrapped.save_pretrained(final_output_dir)
    tokenizer.save_pretrained(final_output_dir)
    print(f"Fine‑tuned model saved to {final_output_dir}")

    
    save_finetuned_model(trainer, output_dir)
    print_finetuning_summary(resource_monitor, moemetrics, train_result, eval_results, perplexity)
    
    return trainer

print("Fine-tuning function defined")

In [ ]:
print("=" * 60)
print("FINE-TUNING DeepSeekMoE (BASELINE)")
print("=" * 60)
fine_tune_model(
    BaselineModel,
    BaselineConfig,
    PRETRAINED_BASELINE,
    OUTPUT_FT_BASELINE,
    is_dynmoe=False
)

In [ ]:
print("=" * 60)
print("FINE-TUNING DYNMoE (BASELINE)")
print("=" * 60)
fine_tune_model(
    DYNMoEBaseModel,
    DYNMoEBaseConfig,
    PRETRAINED_DYNMOE_BASE,
    OUTPUT_FT_DYNMOE_BASE,
    is_dynmoe=True
)

In [ ]:
print("=" * 60)
print("FINE-TUNING DeepSeekMoE DYNMoE Routing")
print("=" * 60)
fine_tune_model(
    DynmoeModel,
    DynmoeConfig,
    PRETRAINED_DYNMOE_ROUTING,
    OUTPUT_FT_DYNMOE_ROUTING,
    is_dynmoe=True
)